# 03) Baseline Models and Feature Selection

In [1]:
import json

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import warnings

from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FEATURE_COLS = columns["feature_cols"]

print(f"train: {train.shape[0]} rows, {len(FEATURE_COLS)} features, target = {TARGET_COL!r}")

train: 8443 rows, 30 features, target = 'target_bps_vs_mid'


## Expanding-window cross-validation split

A single fixed fit/eval split gives one point estimate of eval MSE, entirely dependent on which
handful of days happened to land in eval -- not something to pick a penalty on with confidence.
Expanding-window CV rolls the eval block forward across multiple folds: each fold trains on every
day strictly before its eval block (never after -- no lookahead) and the training window grows as
the boundary moves forward. `data/test.csv` is still untouched; all folds live inside
`data/train.csv`.

Walk-forward, one day at a time: fold 0 trains on the first `MIN_TRAIN_DAYS` days and evals on the
next single day; each later fold folds that eval day into training and evals on the day after it.
`MIN_TRAIN_DAYS` is the warm-up -- how much history a fold needs before it's trusted to fit at all.

In [2]:
unique_dates = np.sort(train["date"].unique())
MIN_TRAIN_DAYS = 7
DELTA_DAYS = 1

folds = []
for fold_idx, eval_start in enumerate(range(MIN_TRAIN_DAYS, len(unique_dates), DELTA_DAYS)):
    fit_dates = unique_dates[:eval_start]
    eval_dates = unique_dates[eval_start:eval_start + DELTA_DAYS]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train["date"].isin(fit_dates)
    eval_mask = train["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  496 rows (1 days, 2025-03-12 to 2025-03-12)
fold 1: fit 3972 rows (8 days, 2025-03-03 to 2025-03-12) | eval  497 rows (1 days, 2025-03-13 to 2025-03-13)
fold 2: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  497 rows (1 days, 2025-03-14 to 2025-03-14)
fold 3: fit 4966 rows (10 days, 2025-03-03 to 2025-03-14) | eval  497 rows (1 days, 2025-03-17 to 2025-03-17)
fold 4: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  496 rows (1 days, 2025-03-18 to 2025-03-18)
fold 5: fit 5959 rows (12 days, 2025-03-03 to 2025-03-18) | eval  497 rows (1 days, 2025-03-19 to 2025-03-19)
fold 6: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  496 rows (1 days, 2025-03-20 to 2025-03-20)
fold 7: fit 6952 rows (14 days, 2025-03-03 to 2025-03-20) | eval  497 rows (1 days, 2025-03-21 to 2025-03-21)
fold 8: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  497 rows (1 days, 2025-03-24 to 2025-03-24)
fold 9: fit 7

## Elastic net, 6x6 grid over alpha and l1_ratio

Ridge -> elastic net: `alpha` (overall penalty strength) and `l1_ratio` (mix between L1/Lasso,
`l1_ratio=1`, and L2/Ridge, `l1_ratio=0`) are tuned jointly on a 6x6 grid, 36 combinations, each
scored the same way as before -- fit on each fold's `fit` side, MSE/R^2/IC on both sides, averaged
across the 10 expanding-window folds. Same pipeline otherwise: median-impute -> standardize ->
model, both fit on `fit` only.

`data/train.csv`/`data/columns.json` now reflect the pruned, 30-feature set from `02)` (raw
non-stationary price levels moved to `IDENTIFIER_COLS`), so this also re-runs the whole sweep
against the corrected features, not just swapping the model.

In [3]:
warnings.filterwarnings("ignore", category=UserWarning)  # l1_ratio=0 -> "use Ridge instead"

alphas = [0.001, 0.01, 0.03, 0.1, 0.3, 1.0]
l1_ratios = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        fit_mses, eval_mses = [], []
        fit_r2s, eval_r2s = [], []
        fit_ics, eval_ics = [], []
        for fit_mask, eval_mask in folds:
            X_fit, y_fit = train.loc[fit_mask, FEATURE_COLS], train.loc[fit_mask, TARGET_COL]
            X_eval, y_eval = train.loc[eval_mask, FEATURE_COLS], train.loc[eval_mask, TARGET_COL]

            model = Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
                ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10_000)),
            ])
            model.fit(X_fit, y_fit)

            pred_fit = model.predict(X_fit)
            pred_eval = model.predict(X_eval)

            fit_mses.append(mean_squared_error(y_fit, pred_fit))
            eval_mses.append(mean_squared_error(y_eval, pred_eval))
            fit_r2s.append(r2_score(y_fit, pred_fit))
            eval_r2s.append(r2_score(y_eval, pred_eval))
            fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
            eval_ics.append(spearmanr(y_eval, pred_eval).statistic)

        results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "fit_mse": np.mean(fit_mses),
            "eval_mse_mean": np.mean(eval_mses),
            "eval_mse_std": np.std(eval_mses),
            "fit_r2": np.mean(fit_r2s),
            "eval_r2": np.mean(eval_r2s),
            "fit_ic": np.mean(fit_ics),
            "eval_ic": np.mean(eval_ics),
        })

results_df = pd.DataFrame(results).sort_values("eval_mse_mean").reset_index(drop=True)
print(results_df.to_string(index=False))

print()
print("eval_mse_mean grid (rows=alpha, cols=l1_ratio):")
pivot = pd.DataFrame(results).pivot(index="alpha", columns="l1_ratio", values="eval_mse_mean")
print(pivot.to_string())

 alpha  l1_ratio   fit_mse  eval_mse_mean  eval_mse_std   fit_r2  eval_r2   fit_ic  eval_ic
 0.100       1.0 13.357665      10.112786      2.608471 0.817995 0.816355 0.824449 0.823747
 0.030       1.0 13.239759      10.193138      2.636631 0.819599 0.814830 0.824845 0.823606
 0.300       1.0 13.626527      10.239783      2.603344 0.814339 0.815678 0.822734 0.821558
 0.030       0.8 13.231727      10.323064      2.661092 0.819707 0.813150 0.824861 0.823452
 0.100       0.8 13.367127      10.380026      2.785595 0.817867 0.814150 0.824949 0.824264
 0.030       0.6 13.233780      10.482041      2.676303 0.819678 0.810731 0.824854 0.823299
 0.010       1.0 13.197808      10.608287      2.469327 0.820161 0.804350 0.824217 0.822295
 0.030       0.4 13.243784      10.657101      2.707535 0.819542 0.808044 0.824761 0.822895
 0.010       0.8 13.195740      10.674069      2.484912 0.820189 0.803251 0.824181 0.822145
 0.100       0.6 13.432972      10.720096      3.070520 0.816978 0.810696 0.8251